# Quantum-Inspired Reservoir Computing
A PyTorch implementation of a Quantum-Inspired Reservoir model utilizing orthogonal matrix decomposition for classification tasks.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# 1. Generate synthetic dataset with non-linear relations
np.random.seed(42)
torch.manual_seed(42)

n_samples = 1000
n_features = 20

X_raw = np.random.randn(n_samples, n_features)
y_raw = (np.sin(X_raw[:, 0]) + np.cos(X_raw[:, 1]) + X_raw[:, 2] > 0).astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_raw, test_size=0.2, random_state=42
)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

In [ ]:
# 2. Define the Quantum-Inspired Reservoir Model (Fixed Dimensions)
class QuantumInspiredReservoir(nn.Module):
    def __init__(self, input_dim, reservoir_dim):
        super(QuantumInspiredReservoir, self).__init__()
        self.reservoir_dim = reservoir_dim
        
        rand_weights = torch.randn(input_dim, reservoir_dim)
        q_weights, _ = torch.qr(rand_weights)
        self.register_buffer('W_res', q_weights)
        
        # The linear readout layer maps from reservoir_dim to 1
        self.readout = nn.Linear(reservoir_dim, 1)
        
    def forward(self, x):
        x_res = torch.tanh(torch.matmul(x, self.W_res))
        out = torch.sigmoid(self.readout(x_res))
        return out

input_dim = n_features
reservoir_dim = 64
model = QuantumInspiredReservoir(input_dim, reservoir_dim)

In [ ]:
# 3. Train the model
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 50
loss_history = []

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())

# 4. Evaluate the model
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t)
    test_preds_binary = (test_preds >= 0.5).float()
    
    acc = accuracy_score(y_test_t.numpy(), test_preds_binary.numpy())
    f1 = f1_score(y_test_t.numpy(), test_preds_binary.numpy(), zero_division=0)

print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")

In [ ]:
# 5. Plot training loss history
plt.figure(figsize=(8, 5))
plt.plot(loss_history, label='Training Loss', color='b', linewidth=2)
plt.title('Quantum-Inspired Reservoir Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss Value')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()